# 05 - 分支式 RAG（多查询检索）

**复杂度：** ⭐⭐⭐

**使用场景：** 多意图查询、跨领域研究、全面主题探索

**核心特性：** 从用户问题生成多个子查询，并行检索以获得更好的覆盖范围。

**示例：**
```
查询："比较 OpenAI 和 HuggingFace 嵌入模型的成本和性能"

生成的子查询：
1. "OpenAI 嵌入模型的定价和成本"
2. "HuggingFace 嵌入模型的性能基准测试"
3. "嵌入模型提供商的比较"

→ 检索涵盖所有方面的多样化文档
```

## 1. 环境设置

In [1]:
import sys
sys.path.append('../..')

from shared.config import create_chat_model, create_embeddings, DEFAULT_MODEL, DEFAULT_TEMPERATURE, DEFAULT_VISION_MODEL
from shared.config import HF_VECTOR_STORE_PATH, DEFAULT_MODEL
from shared.utils import load_vector_store, print_section_header, format_docs
from shared.prompts import RAG_PROMPT_TEMPLATE, MULTI_QUERY_PROMPT

print_section_header("Setup: Branched RAG")

embeddings = create_embeddings()
vectorstore = load_vector_store(HF_VECTOR_STORE_PATH, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
llm = create_chat_model(model=DEFAULT_MODEL, temperature=0)

print("✅ Setup complete!")


SETUP: BRANCHED RAG



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded vector store from d:\ZKS_Data\AI\langchain-rag-tutorial\notebooks\advanced_architectures\..\..\data\vector_stores\huggingface_embeddings


langchain-openai injected a custom httpx transport to apply `http_socket_options`, which disables httpx's proxy auto-detection (system proxy configuration detected). Set `LANGCHAIN_OPENAI_TCP_KEEPALIVE=0` or pass `http_socket_options=()` to restore default proxy behavior, or supply `openai_proxy` / your own `http_client` / `http_async_client` to take full control.


✅ Setup complete!


## 2. 多查询检索器

使用 `MultiQueryRetriever` 生成替代查询并检索文档。

In [2]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
import logging

print_section_header("Multi-Query Retriever")

# Custom multi-query retriever implementation (no legacy dependencies)
class CustomMultiQueryRetriever:
    """Custom multi-query retriever that generates multiple query variations."""
    
    def __init__(self, base_retriever, llm, prompt):
        self.base_retriever = base_retriever
        self.llm = llm
        self.prompt = prompt
        self.logger = logging.getLogger("custom.multi_query")
        
    def invoke(self, query: str) -> list:
        """Retrieve documents using multiple query variations."""
        # Generate alternative queries
        query_gen_chain = self.prompt | self.llm | StrOutputParser()
        generated = query_gen_chain.invoke({"question": query})
        
        # Parse generated queries (one per line)
        alternative_queries = [q.strip() for q in generated.split('\n') if q.strip()]
        all_queries = [query] + alternative_queries[:3]  # Original + top 3 alternatives
        
        self.logger.info(f"Generated queries: {all_queries}")
        
        # Retrieve documents for each query
        all_docs = []
        for q in all_queries:
            docs = self.base_retriever.invoke(q)
            all_docs.extend(docs)
        
        # Deduplicate by content
        unique_docs = []
        seen_content = set()
        for doc in all_docs:
            content_hash = doc.page_content[:200]  # Use first 200 chars for dedup
            if content_hash not in seen_content:
                unique_docs.append(doc)
                seen_content.add(content_hash)
        
        return unique_docs

# Enable logging to see generated queries
logging.basicConfig(level=logging.WARNING)
logging.getLogger("custom.multi_query").setLevel(logging.INFO)

# Create custom multi-query retriever
multi_query_retriever = CustomMultiQueryRetriever(
    base_retriever=retriever,
    llm=llm,
    prompt=MULTI_QUERY_PROMPT
)

print("✓ Custom MultiQueryRetriever created")
print("  - Generates 3 alternative queries")
print("  - Retrieves documents for each")
print("  - Deduplicates results")
print("  - No legacy dependencies!")


MULTI-QUERY RETRIEVER

✓ Custom MultiQueryRetriever created
  - Generates 3 alternative queries
  - Retrieves documents for each
  - Deduplicates results
  - No legacy dependencies!


## 3. 测试多查询检索

In [3]:
from shared.utils import print_results

print_section_header("Multi-Query Test")

query = "How to optimize RAG system performance?"
print(f"Original Query: '{query}'\n")

# Retrieve with multi-query
docs = multi_query_retriever.invoke(query)

print(f"\n✓ Retrieved {len(docs)} unique documents")
print_results(docs, "Multi-Query Results", max_docs=4, preview_length=150)


MULTI-QUERY TEST

Original Query: 'How to optimize RAG system performance?'



INFO:custom.multi_query:Generated queries: ['How to optimize RAG system performance?', 'What are the best techniques for improving retrieval-augmented generation efficiency?', 'How can I fine-tune a RAG pipeline for better accuracy and latency?', 'What strategies enhance the quality and speed of a RAG system?']



✓ Retrieved 8 unique documents

Multi-Query Results
--------------------------------------------------------------------------------

1. Source: https://python.langchain.com/docs/use_cases/question_answering/
   Type: local_text_download
   Date: 2026-05-09
   Content: Integrations
: 40+ integrations to choose from.
Interface
: API reference for the base interface.
This completes the
Indexing
portion of the pipeline....

2. Source: https://python.langchain.com/docs/modules/model_io/llms/
   Type: local_text_download
   Date: 2026-05-09
   Content: batch_as_completed()
, results may arrive out of order. Each includes the input index for matching to reconstruct the original order as needed.
When p...

3. Source: https://python.langchain.com/docs/modules/model_io/llms/
   Type: local_text_download
   Date: 2026-05-09
   Content: any
tool from a given list:
Force use of any tool
Force use of specific tools
model_with_tools
=
model
.
bind_tools
([
tool_1
],
tool_choice
=
"any"
)...

4. Sou

## 4. 构建分支式 RAG 链

In [4]:
print_section_header("Branched RAG Chain")

def multi_query_retrieve(query: str):
    """Wrapper function for multi-query retrieval."""
    return multi_query_retriever.invoke(query)

branched_chain = (
    {"context": RunnableLambda(multi_query_retrieve) | format_docs, "input": RunnablePassthrough()}
    | RAG_PROMPT_TEMPLATE
    | llm
    | StrOutputParser()
)

print("✓ Branched RAG chain created")

# Test
query = "What are the differences between similarity and MMR retrieval?"
print(f"\nQuery: '{query}'\n")
print("=" * 80)

response = branched_chain.invoke(query)
print(response)

print("\n" + "=" * 80)
print("\n✅ Branched RAG provides more comprehensive answers!")


BRANCHED RAG CHAIN

✓ Branched RAG chain created

Query: 'What are the differences between similarity and MMR retrieval?'



INFO:custom.multi_query:Generated queries: ['What are the differences between similarity and MMR retrieval?', 'How do similarity-based search and maximum marginal relevance (MMR) retrieval differ in their approaches and outputs?', 'What distinguishes cosine similarity retrieval from MMR (diversity-focused) retrieval methods?', 'Can you compare and contrast standard similarity retrieval with MMR retrieval, particularly regarding relevance vs. diversity trade-offs?']


The provided context does not contain any information about the differences between similarity retrieval and MMR (Maximal Marginal Relevance) retrieval. It only describes constructing a RAG agent with a retrieval tool, indexing, and generation, but does not compare or define these specific retrieval methods. Therefore, I cannot answer your question based on the given context.


✅ Branched RAG provides more comprehensive answers!


## 5. 对比：简单 RAG vs 分支式 RAG

In [5]:
print_section_header("Comparison")

# Simple RAG
simple_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | RAG_PROMPT_TEMPLATE
    | llm
    | StrOutputParser()
)

query = "How to implement embeddings in RAG?"

print(f"Query: '{query}'\n")
print("[SIMPLE RAG]")
simple_docs = retriever.invoke(query)
print(f"Documents retrieved: {len(simple_docs)}")

print("\n[BRANCHED RAG]")
branched_docs = multi_query_retriever.invoke(query)
print(f"Documents retrieved: {len(branched_docs)} (from multiple queries)")

print("\n💡 Branched RAG typically retrieves more diverse documents")


COMPARISON

Query: 'How to implement embeddings in RAG?'

[SIMPLE RAG]
Documents retrieved: 4

[BRANCHED RAG]


INFO:custom.multi_query:Generated queries: ['How to implement embeddings in RAG?', 'What are the steps to incorporate embeddings into a Retrieval-Augmented Generation pipeline?', 'How to generate and use vector embeddings for document retrieval in RAG systems?', 'What are the best practices for integrating embedding models in RAG architecture?']


Documents retrieved: 6 (from multiple queries)

💡 Branched RAG typically retrieves more diverse documents


## 6. 分析分支式 RAG 的详细耗时

下面分别对简单 RAG 和分支式 RAG 做分阶段计时。

简单 RAG 的阶段包括：
- 检索
- 上下文整理
- 提示词构建
- 模型生成
- 输出解析

分支式 RAG 额外增加两个阶段：
- 子查询生成
- 多轮检索后的去重

为了减少偶然波动，这里对每种链路重复运行多次并计算平均耗时。

In [6]:
from statistics import mean
from time import perf_counter

from shared.utils import print_comparison_table

parser = StrOutputParser()
profile_query = "How can branched retrieval improve answer coverage in a RAG system?"
profile_runs = 3


def deduplicate_docs(all_docs):
    unique_docs = []
    seen_content = set()
    for doc in all_docs:
        content_hash = doc.page_content[:200]
        if content_hash not in seen_content:
            unique_docs.append(doc)
            seen_content.add(content_hash)
    return unique_docs


def profile_simple_rag(query, runs=3):
    measurements = []
    final_docs = []
    final_answer = ''

    for _ in range(runs):
        t0 = perf_counter()
        docs = retriever.invoke(query)
        t1 = perf_counter()

        context = format_docs(docs)
        t2 = perf_counter()

        prompt_value = RAG_PROMPT_TEMPLATE.invoke({"context": context, "input": query})
        t3 = perf_counter()

        llm_response = llm.invoke(prompt_value)
        t4 = perf_counter()

        answer = parser.invoke(llm_response)
        t5 = perf_counter()

        measurements.append(
            {
                "query_generation_ms": 0.0,
                "retrieval_ms": (t1 - t0) * 1000,
                "dedup_ms": 0.0,
                "format_ms": (t2 - t1) * 1000,
                "prompt_ms": (t3 - t2) * 1000,
                "llm_ms": (t4 - t3) * 1000,
                "parser_ms": (t5 - t4) * 1000,
                "total_ms": (t5 - t0) * 1000,
            }
        )
        final_docs = docs
        final_answer = answer

    averages = {
        key: mean(item[key] for item in measurements)
        for key in measurements[0]
    }
    return averages, final_docs, final_answer


def profile_branched_rag(query, runs=3):
    measurements = []
    final_docs = []
    final_queries = []
    final_answer = ''

    for _ in range(runs):
        t0 = perf_counter()
        query_gen_chain = MULTI_QUERY_PROMPT | llm | StrOutputParser()
        generated = query_gen_chain.invoke({"question": query})
        t1 = perf_counter()

        alternative_queries = [q.strip().lstrip('-').strip() for q in generated.split('\n') if q.strip()]
        all_queries = [query] + alternative_queries[:3]

        all_docs = []
        for generated_query in all_queries:
            docs = retriever.invoke(generated_query)
            all_docs.extend(docs)
        t2 = perf_counter()

        unique_docs = deduplicate_docs(all_docs)
        t3 = perf_counter()

        context = format_docs(unique_docs)
        t4 = perf_counter()

        prompt_value = RAG_PROMPT_TEMPLATE.invoke({"context": context, "input": query})
        t5 = perf_counter()

        llm_response = llm.invoke(prompt_value)
        t6 = perf_counter()

        answer = parser.invoke(llm_response)
        t7 = perf_counter()

        measurements.append(
            {
                "query_generation_ms": (t1 - t0) * 1000,
                "retrieval_ms": (t2 - t1) * 1000,
                "dedup_ms": (t3 - t2) * 1000,
                "format_ms": (t4 - t3) * 1000,
                "prompt_ms": (t5 - t4) * 1000,
                "llm_ms": (t6 - t5) * 1000,
                "parser_ms": (t7 - t6) * 1000,
                "total_ms": (t7 - t0) * 1000,
            }
        )
        final_docs = unique_docs
        final_queries = all_queries
        final_answer = answer

    averages = {
        key: mean(item[key] for item in measurements)
        for key in measurements[0]
    }
    return averages, final_docs, final_queries, final_answer


def percent(part, total):
    if not total:
        return '0.0%'
    return f'{part / total * 100:.1f}%'


print_section_header("Timing Experiment: Simple vs Branched RAG")
print(f"测试查询: {profile_query}")
print(f"每种链路重复次数: {profile_runs}")

simple_stats, simple_profile_docs, simple_profile_answer = profile_simple_rag(profile_query, runs=profile_runs)
branched_stats, branched_profile_docs, branched_profile_queries, branched_profile_answer = profile_branched_rag(profile_query, runs=profile_runs)

table_rows = []
for pipeline_name, stats in [("简单 RAG", simple_stats), ("分支式 RAG", branched_stats)]:
    total_ms = stats["total_ms"]
    table_rows.extend(
        [
            [pipeline_name, "子查询生成", f"{stats['query_generation_ms']:.2f}", percent(stats['query_generation_ms'], total_ms)],
            [pipeline_name, "检索", f"{stats['retrieval_ms']:.2f}", percent(stats['retrieval_ms'], total_ms)],
            [pipeline_name, "去重", f"{stats['dedup_ms']:.2f}", percent(stats['dedup_ms'], total_ms)],
            [pipeline_name, "上下文整理", f"{stats['format_ms']:.2f}", percent(stats['format_ms'], total_ms)],
            [pipeline_name, "提示词构建", f"{stats['prompt_ms']:.2f}", percent(stats['prompt_ms'], total_ms)],
            [pipeline_name, "模型生成", f"{stats['llm_ms']:.2f}", percent(stats['llm_ms'], total_ms)],
            [pipeline_name, "输出解析", f"{stats['parser_ms']:.2f}", percent(stats['parser_ms'], total_ms)],
            [pipeline_name, "总计", f"{stats['total_ms']:.2f}", "100.0%"],
        ]
    )

print("\n平均耗时明细 (ms):")
print_comparison_table(
    table_rows,
    headers=["链路", "阶段", "平均耗时(ms)", "总耗时占比"],
)

print("\n" + "=" * 80)
print("关键观察")
print("=" * 80)
print(f"简单 RAG 命中文档数: {len(simple_profile_docs)}")
print(f"分支式 RAG 命中文档数: {len(branched_profile_docs)}")
print(f"分支式 RAG 使用的查询数: {len(branched_profile_queries)}")
print(f"简单 RAG 平均总耗时: {simple_stats['total_ms']:.2f} ms")
print(f"分支式 RAG 平均总耗时: {branched_stats['total_ms']:.2f} ms")
print(f"分支式 RAG 的额外子查询生成耗时占比: {percent(branched_stats['query_generation_ms'], branched_stats['total_ms'])}")
print(f"分支式 RAG 的多轮检索耗时占比: {percent(branched_stats['retrieval_ms'], branched_stats['total_ms'])}")

print("\n分支式 RAG 最后一次运行使用的查询:")
for idx, generated_query in enumerate(branched_profile_queries, start=1):
    print(f"{idx}. {generated_query}")

print("\n答案预览（简单 RAG）:")
print(simple_profile_answer[:250])
if len(simple_profile_answer) > 250:
    print("...")

print("\n答案预览（分支式 RAG）:")
print(branched_profile_answer[:250])
if len(branched_profile_answer) > 250:
    print("...")



TIMING EXPERIMENT: SIMPLE VS BRANCHED RAG

测试查询: How can branched retrieval improve answer coverage in a RAG system?
每种链路重复次数: 3

平均耗时明细 (ms):
链路       阶段     平均耗时(ms)  总耗时占比   
----------------------------------
简单 RAG   子查询生成  0.00      0.0%    
简单 RAG   检索     9.76      0.2%    
简单 RAG   去重     0.00      0.0%    
简单 RAG   上下文整理  0.01      0.0%    
简单 RAG   提示词构建  0.36      0.0%    
简单 RAG   模型生成   4980.70   99.8%   
简单 RAG   输出解析   0.16      0.0%    
简单 RAG   总计     4990.98   100.0%  
分支式 RAG  子查询生成  3989.26   20.3%   
分支式 RAG  检索     34.52     0.2%    
分支式 RAG  去重     0.02      0.0%    
分支式 RAG  上下文整理  0.01      0.0%    
分支式 RAG  提示词构建  0.33      0.0%    
分支式 RAG  模型生成   15649.63  79.5%   
分支式 RAG  输出解析   0.15      0.0%    
分支式 RAG  总计     19673.92  100.0%  

关键观察
简单 RAG 命中文档数: 4
分支式 RAG 命中文档数: 4
分支式 RAG 使用的查询数: 4
简单 RAG 平均总耗时: 4990.98 ms
分支式 RAG 平均总耗时: 19673.92 ms
分支式 RAG 的额外子查询生成耗时占比: 20.3%
分支式 RAG 的多轮检索耗时占比: 0.2%

分支式 RAG 最后一次运行使用的查询:
1. How can branched retrieval improve answe

## 总结

**流程：**
```
用户查询 → 生成子查询 → 并行检索 → 去重 → LLM → 响应
```

**优势：**
✅ 对复杂查询有更好的覆盖范围  
✅ 捕获多个方面  
✅ 更多样化的视角  
✅ 处理多意图问题  

**局限性：**
- 更高的延迟（多次检索）
- 更多的 API 调用（成本）
- 可能检索到冗余信息

**本 notebook 新增：**
- 简单 RAG 与分支式 RAG 的详细耗时实验
- 对子查询生成、多轮检索、去重、提示构建、模型生成等阶段的拆分统计

**适用场景：**
- 广泛的研究问题
- 多概念查询
- 需要全面覆盖的情况

**下一步：** [06_hyde.ipynb](06_hyde.ipynb) - 假设性文档嵌入